In [30]:
# =============================================================================
# CONFIG — Serie A xT Analysis (Opta CSVs)
# =============================================================================
# 1) DATA_DIR points to your match-event CSV folder.
#    It tries the project-local path first, then your Documents folder.
# 2) Change FILE_NAME to analyse a different match.
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---- notebook root (two levels up from notebooks/) --------------------------
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# ---- data directory candidates (first existing wins) ------------------------
_DATA_DIR_CANDIDATES = [
    # project-local (preferred)
    PROJECT_ROOT / "data" / "raw" / "serie_a_2025_2026" / "events",
    # user Documents paths (legacy)
    Path.home() / "Documents" / "Master Degree in Sport Analytics"
    / "2. Management and Architecture of Sports Database" / "Result_SerieA_25_26",
    Path.home() / "Documents" / "Master Degree in Sport Analytics"
    / "2. Management and Architecture of Sports Database" / "selenium wire" / "Result_SerieA_25_26",
]

DATA_DIR: Path | None = None
for _d in _DATA_DIR_CANDIDATES:
    if _d.exists():
        DATA_DIR = _d
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        f"No data directory found. Tried:\n" + "\n".join(f"  • {d}" for d in _DATA_DIR_CANDIDATES)
    )

print("DATA_DIR:", DATA_DIR)

# ---- pick ONE match file ----------------------------------------------------
FILE_NAME = "27_Pisa_Bologna_btdw5nixu80jqgu17y4qmgb8 copia.csv"
FILE_PATH = DATA_DIR / FILE_NAME


# ---- robust CSV loader ------------------------------------------------------
def load_match_csv(file_path: Path) -> pd.DataFrame:
    """Load an Opta-style match CSV, handling delimiter quirks."""
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    # Try comma first, then semicolon
    try:
        df = pd.read_csv(file_path)
    except Exception:
        df = pd.read_csv(file_path, sep=";", engine="python")

    # If everything ended up in one column, try common separators
    if df.shape[1] == 1:
        col0 = df.columns[0]
        for sep in [",", ";", "\t", "|"]:
            tmp = pd.read_csv(file_path, sep=sep, engine="python")
            if tmp.shape[1] > 1:
                df = tmp
                break

        # Last resort: split the single string column
        if df.shape[1] == 1 and df[col0].astype(str).str.contains(";").any():
            df = df[col0].str.split(";", expand=True)
            df.columns = df.iloc[0]
            df = df.iloc[1:].reset_index(drop=True)

    return df


df = load_match_csv(FILE_PATH)
print(f"Loaded: {FILE_NAME}  ({len(df)} rows × {df.shape[1]} cols)")
display(df.head())

DATA_DIR: /Users/ricki/Local Projects/FMP_SerieA_Dashboard/data/raw/serie_a_2025_2026/events
Loaded: 27_Pisa_Bologna_btdw5nixu80jqgu17y4qmgb8 copia.csv  (1700 rows × 248 cols)


,general_id,event_id,event,type_id,period_id,time_min,time_sec,contestant_id,team_name,team_code,...,Post match complete,competition_id,competition_name,competition_known_name,competition_sponsor_name,competition_code,venue_id,venue_long_name,formation,position
0,2906647553,1,Team setp up,34,16,0,0,bfztlrr6176xsdt2bmp0u1d2d,Pisa Sporting Club,PIS,...,NaN,1r097lpxe0xn03ihb7wi98kao,Serie A,Italian Serie A,Serie A Enilive,SEA,bohw04nmt77yy9hdyycxgk070,Arena Garibaldi - Stadio Romeo Anconetani,352,NaN
1,2906647699,1,Team setp up,34,16,0,0,ej5er0oyngdw138yuumwqbyqt,Bologna FC 1909,BOL,...,NaN,1r097lpxe0xn03ihb7wi98kao,Serie A,Italian Serie A,Serie A Enilive,SEA,bohw04nmt77yy9hdyycxgk070,Arena Garibaldi - Stadio Romeo Anconetani,433,NaN
2,2906651037,2,Start,32,1,0,0,bfztlrr6176xsdt2bmp0u1d2d,Pisa Sporting Club,PIS,...,NaN,1r097lpxe0xn03ihb7wi98kao,Serie A,Italian Serie A,Serie A Enilive,SEA,bohw04nmt77yy9hdyycxgk070,Arena Garibaldi - Stadio Romeo Anconetani,352,NaN
3,2906651035,2,Start,32,1,0,0,ej5er0oyngdw138yuumwqbyqt,Bologna FC 1909,BOL,...,NaN,1r097lpxe0xn03ihb7wi98kao,Serie A,Italian Serie A,Serie A Enilive,SEA,bohw04nmt77yy9hdyycxgk070,Arena Garibaldi - Stadio Romeo Anconetani,433,NaN
4,2906651045,3,Pass,1,1,0,0,ej5er0oyngdw138yuumwqbyqt,Bologna FC 1909,BOL,...,NaN,1r097lpxe0xn03ihb7wi98kao,Serie A,Italian Serie A,Serie A Enilive,SEA,bohw04nmt77yy9hdyycxgk070,Arena Garibaldi - Stadio Romeo Anconetani,433,CF


In [31]:
# =============================================================================
# TEAM DETECTION & SELECTION
# =============================================================================

def detect_team_col(df: pd.DataFrame) -> str:
    """Find the column that holds team names."""
    for c in ["team_name", "team", "teamName", "team_code", "teamId", "team_id"]:
        if c in df.columns:
            return c
    raise KeyError(f"No team column found. Available: {list(df.columns)}")

TEAM_COL = detect_team_col(df)
teams = df[TEAM_COL].dropna().astype(str).value_counts().index.tolist()
TEAM_A = teams[0] if len(teams) > 0 else None
TEAM_B = teams[1] if len(teams) > 1 else None

print(f"TEAM_COL: {TEAM_COL}")
print(f"Teams in match: {TEAM_A}  vs  {TEAM_B}")

# ---- Choose which team to analyse (change only this) ------------------------
TEAM_TO_ANALYSE = TEAM_A

team_df = df[df[TEAM_COL].astype(str) == str(TEAM_TO_ANALYSE)].copy()
print(f"\nAnalysing: {TEAM_TO_ANALYSE}  ({len(team_df)} events)")

TEAM_COL: team_name
Teams in match: Bologna FC 1909  vs  Pisa Sporting Club

Analysing: Bologna FC 1909  (950 events)


In [32]:
# =============================================================================
# COLUMN DETECTION  &  HELPER FUNCTIONS
# =============================================================================
# All reusable utilities live here so later cells stay concise.

# ---- generic column finder --------------------------------------------------
def _first_existing(cols, candidates):
    """Return the first candidate name that exists in *cols*."""
    col_set = set(cols)
    for c in candidates:
        if c in col_set:
            return c
    return None


# ---- detect coordinate columns ----------------------------------------------
X_COL    = _first_existing(df.columns, ["x", "X", "start_x", "startX", "pos_x", "x_coordinate"])
Y_COL    = _first_existing(df.columns, ["y", "Y", "start_y", "startY", "pos_y", "y_coordinate"])
ENDX_COL = _first_existing(df.columns, ["endX", "end_x", "x_end", "endX_coordinate", "Pass End X", "carry_end_x"])
ENDY_COL = _first_existing(df.columns, ["endY", "end_y", "y_end", "endY_coordinate", "Pass End Y", "carry_end_y"])

# ---- detect event-type & outcome columns ------------------------------------
TYPE_COL    = _first_existing(df.columns, ["type_name", "type", "event", "event_type", "eventType", "eventName"])
OUTCOME_COL = _first_existing(df.columns, ["outcome_name", "outcome", "outcomeType", "result", "is_success", "success"])

print("Detected columns:")
print(f"  TEAM : {TEAM_COL}")
print(f"  TYPE : {TYPE_COL}")
print(f"  OUTC : {OUTCOME_COL}")
print(f"  X/Y  : {X_COL}, {Y_COL}")
print(f"  EndXY: {ENDX_COL}, {ENDY_COL}")

if X_COL is None or Y_COL is None:
    raise KeyError("Could not detect x/y coordinate columns. Check df.columns.")


# ---- numeric coercion helper ------------------------------------------------
def as_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


# ---- pass/cross mask --------------------------------------------------------
def guess_pass_mask(d: pd.DataFrame, type_col: str | None = None) -> pd.Series:
    """Boolean mask for pass-like actions (including crosses)."""
    _tcol = type_col if type_col is not None else TYPE_COL
    if _tcol is not None and _tcol in d.columns:
        t = d[_tcol].astype(str).str.lower()
        base = t.str.contains("pass") | t.str.contains("cross")
    else:
        base = pd.Series(False, index=d.index)
        for c in d.columns:
            s = d[c].astype(str).str.lower()
            if s.str.contains("pass").any() or s.str.contains("cross").any():
                base = s.str.contains("pass") | s.str.contains("cross")
                break

    # Dedicated "Cross" qualifier column (common in Opta CSVs)
    cross_cols = [c for c in d.columns if str(c).strip().lower() in ("cross",) or " cross" in str(c).strip().lower()]
    if cross_cols:
        cs = d[cross_cols[0]]
        s_num = pd.to_numeric(cs, errors="coerce")
        if s_num.notna().mean() > 0.5:
            cross_q = s_num == 1
        elif pd.api.types.is_bool_dtype(cs):
            cross_q = cs.fillna(False)
        else:
            cross_q = cs.astype(str).str.strip().str.lower().isin({"1", "true", "t", "yes", "y"})
        base = base | cross_q

    return base


# ---- success/outcome column detection & mask --------------------------------
def guess_success_col(d: pd.DataFrame) -> str | None:
    """Find a column encoding event outcome/success."""
    candidates = [
        "outcomeType", "outcome_type", "outcome", "Outcome",
        "result", "Result",
        "isSuccessful", "is_successful", "successful", "success",
        "accurate", "isAccurate", "is_accurate",
        "won", "completed", "Complete", "completedPass",
        "Pass Outcome", "PassOutcome", "pass_outcome",
        "Cross Outcome", "cross_outcome",
    ]
    c = _first_existing(d.columns, candidates)
    if c is not None:
        return c
    # fuzzy fallback
    for col in d.columns:
        name = str(col).strip().lower()
        if any(k in name for k in ["outcome", "result", "success", "accurate", "complete"]):
            return col
    return None


def mask_success(d: pd.DataFrame, col: str) -> pd.Series:
    """Boolean mask for successful/completed events."""
    s = d[col]

    # boolean dtype
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False)

    # numeric 0/1
    s_num = pd.to_numeric(s, errors="coerce")
    if s_num.notna().mean() > 0.5:
        return s_num == 1

    # string-based
    s_str = s.astype(str).str.strip().str.lower()
    good = {"1", "true", "t", "yes", "y", "successful", "success", "complete", "completed", "accurate", "won"}
    return s_str.isin(good)


# ---- coordinate flipping (ensure L→R attack direction) ----------------------
def flip_to_ltr(df: pd.DataFrame, x0="x0", y0="y0", x1="x1", y1="y1", scale=100.0) -> pd.DataFrame:
    """Flip actions moving right→left so all actions go left→right."""
    m = df[x1] < df[x0]
    if m.any():
        df.loc[m, [x0, x1]] = scale - df.loc[m, [x0, x1]].values
        df.loc[m, [y0, y1]] = scale - df.loc[m, [y0, y1]].values
    return df


# ---- xT binning helper ------------------------------------------------------
def bin_xy(x, y, bins_x=12, bins_y=8, scale=100.0):
    """Map (x, y) coordinates to grid bin indices."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ix = np.floor(np.clip(x, 0, scale - 1e-6) / (scale / bins_x)).astype(int)
    iy = np.floor(np.clip(y, 0, scale - 1e-6) / (scale / bins_y)).astype(int)
    return ix, iy


# ---- pitch layout -----------------------------------------------------------
def pitch_layout(fig, title, *, width=900, height=600):
    """Apply a 0-100 Opta pitch overlay to a Plotly figure."""
    fig.update_xaxes(range=[0, 100], showgrid=False, zeroline=False, visible=False)
    fig.update_yaxes(range=[0, 100], showgrid=False, zeroline=False, visible=False,
                     scaleanchor="x", scaleratio=0.68, autorange=False)
    fig.update_layout(title=title, width=width, height=height,
                      margin=dict(l=20, r=20, t=60, b=20))
    # outline + halfway
    fig.add_shape(type="rect", x0=0, y0=0, x1=100, y1=100, line=dict(width=2))
    fig.add_shape(type="line", x0=50, y0=0, x1=50, y1=100, line=dict(width=1))
    # penalty areas
    fig.add_shape(type="rect", x0=0, y0=21, x1=17, y1=79, line=dict(width=1))
    fig.add_shape(type="rect", x0=83, y0=21, x1=100, y1=79, line=dict(width=1))
    # 6-yard boxes
    fig.add_shape(type="rect", x0=0, y0=36, x1=6, y1=64, line=dict(width=1))
    fig.add_shape(type="rect", x0=94, y0=36, x1=100, y1=64, line=dict(width=1))
    # centre circle (approx)
    r = 9.15
    fig.add_shape(type="circle", x0=50 - r, y0=50 - r * (100 / 68),
                  x1=50 + r, y1=50 + r * (100 / 68), line=dict(width=1))
    return fig


# ---- pitch shapes for subplots (needs row/col args) -------------------------
def add_pitch_shapes(fig, row=1, col=1):
    """Draw pitch lines on a subplot panel."""
    kw = dict(row=row, col=col)
    fig.add_shape(type="rect", x0=0, y0=0, x1=100, y1=100,
                  line=dict(width=0), fillcolor="#e9eff7", layer="below", **kw)
    fig.add_shape(type="rect", x0=0, y0=0, x1=100, y1=100, line=dict(width=2, color="black"), **kw)
    fig.add_shape(type="line", x0=50, y0=0, x1=50, y1=100, line=dict(width=1, color="black"), **kw)
    fig.add_shape(type="rect", x0=0, y0=21, x1=17, y1=79, line=dict(width=1, color="black"), **kw)
    fig.add_shape(type="rect", x0=83, y0=21, x1=100, y1=79, line=dict(width=1, color="black"), **kw)
    fig.add_shape(type="rect", x0=0, y0=36, x1=6, y1=64, line=dict(width=1, color="black"), **kw)
    fig.add_shape(type="rect", x0=94, y0=36, x1=100, y1=64, line=dict(width=1, color="black"), **kw)
    r = 9.15
    fig.add_shape(type="circle", x0=50 - r, y0=50 - r * (100 / 68),
                  x1=50 + r, y1=50 + r * (100 / 68), line=dict(width=1, color="black"), **kw)
    return fig

Detected columns:
  TEAM : team_name
  TYPE : event
  OUTC : outcome
  X/Y  : x, y
  EndXY: Pass End X, Pass End Y


In [33]:
# Quick sanity check: coordinate ranges (should be ~0–100 for Opta)
for c in [X_COL, Y_COL, ENDX_COL, ENDY_COL]:
    if c is not None:
        s = pd.to_numeric(df[c], errors="coerce")
        print(f"  {c:12s}  min={s.min():.1f}  max={s.max():.1f}")

  x             min=-1.6  max=101.5
  y             min=-1.8  max=101.8
  Pass End X    min=0.0  max=100.0
  Pass End Y    min=0.0  max=100.0


In [34]:
# =============================================================================
# xT GRID (12 × 8) — static expected-threat values
# =============================================================================
# Rows = y-bins (0=bottom → 7=top), Cols = x-bins (0=own goal → 11=opp. goal).
# Values increase near the opponent's goal.

XT_GRID_12x8 = np.array([
    [0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.001, 0.001, 0.002, 0.003, 0.004, 0.006],
    [0.000, 0.000, 0.000, 0.000, 0.000, 0.001, 0.001, 0.002, 0.003, 0.004, 0.006, 0.009],
    [0.000, 0.000, 0.000, 0.000, 0.001, 0.001, 0.002, 0.003, 0.004, 0.006, 0.009, 0.013],
    [0.000, 0.000, 0.000, 0.001, 0.001, 0.002, 0.003, 0.004, 0.006, 0.009, 0.013, 0.018],
    [0.000, 0.000, 0.001, 0.001, 0.002, 0.003, 0.004, 0.006, 0.009, 0.013, 0.018, 0.024],
    [0.000, 0.001, 0.001, 0.002, 0.003, 0.004, 0.006, 0.009, 0.013, 0.018, 0.024, 0.032],
    [0.001, 0.001, 0.002, 0.003, 0.004, 0.006, 0.009, 0.013, 0.018, 0.024, 0.032, 0.043],
    [0.001, 0.002, 0.003, 0.004, 0.006, 0.009, 0.013, 0.018, 0.024, 0.032, 0.043, 0.057],
], dtype=float)

# Symmetrise across the horizontal midline to avoid top/bottom bias
XT_SYM = (XT_GRID_12x8 + np.flipud(XT_GRID_12x8)) / 2


def compute_xt_added(actions: pd.DataFrame, xt_grid=XT_SYM) -> pd.DataFrame:
    """
    Given a DataFrame with columns x0, y0, x1, y1 (0-100 Opta coords, L→R),
    compute xT_start, xT_end, and xT_added.  Returns the same DataFrame with
    three new columns added.
    """
    ix0, iy0 = bin_xy(actions["x0"].values, actions["y0"].values)
    ix1, iy1 = bin_xy(actions["x1"].values, actions["y1"].values)
    actions["xT_start"] = xt_grid[iy0, ix0]
    actions["xT_end"]   = xt_grid[iy1, ix1]
    actions["xT_added"] = actions["xT_end"] - actions["xT_start"]
    return actions

In [35]:
# =============================================================================
# SINGLE-MATCH xT ANALYSIS — successful passes/crosses with positive xT
# =============================================================================

# ---- 1) Filter to pass-like actions for the selected team -------------------
passes = team_df[guess_pass_mask(team_df)].copy()

# ---- 2) Keep only successful / completed actions ----------------------------
SUCCESS_COL = guess_success_col(passes)
if SUCCESS_COL is None:
    print("⚠️  No success/outcome column detected — keeping ALL passes/crosses.")
else:
    ok = mask_success(passes, SUCCESS_COL)
    n_before = len(passes)
    passes = passes[ok].copy()
    print(f"Success column: {SUCCESS_COL}  →  kept {len(passes)}/{n_before} successful actions")

# ---- 3) Normalise coordinates & flip to L→R ---------------------------------
if ENDX_COL is None or ENDY_COL is None:
    raise KeyError("End-coordinate columns (endX / endY) not detected. xT requires them.")

passes["x0"] = as_numeric(passes[X_COL]).clip(0, 100)
passes["y0"] = as_numeric(passes[Y_COL]).clip(0, 100)
passes["x1"] = as_numeric(passes[ENDX_COL]).clip(0, 100)
passes["y1"] = as_numeric(passes[ENDY_COL]).clip(0, 100)
passes = passes.dropna(subset=["x0", "y0", "x1", "y1"])
passes = flip_to_ltr(passes)

# ---- 4) Compute xT ---------------------------------------------------------
passes = compute_xt_added(passes)

# ---- 5) Keep only positive xT actions (> 0) ---------------------------------
prog = passes[passes["xT_added"] > 0].copy()

# Player label for ranking
PLAYER_COL = _first_existing(passes.columns, ["player_name", "player", "playerName", "Player"])
if PLAYER_COL is None:
    prog["player_label"] = "unknown"
    print("⚠️  No player-name column detected.")
else:
    prog["player_label"] = prog[PLAYER_COL].astype(str)

print(f"\nCompleted passes: {len(passes)}  |  Positive xT: {len(prog)}"
      f"  |  Total xT added: {prog['xT_added'].sum():.4f}")

# ---- 6) Player xT ranking (top 10) -----------------------------------------
rank_df = (
    prog.groupby("player_label", as_index=False)["xT_added"]
        .sum()
        .sort_values("xT_added", ascending=False)
        .head(10)
)

# ---- 7) Visualisation: pitch arrows + player ranking bar chart --------------
TOP_N = 80
topN = prog.sort_values("xT_added", ascending=False).head(TOP_N)

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.78, 0.22],
    horizontal_spacing=0.03,
    specs=[[{"type": "xy"}, {"type": "xy"}]],
)

# Pitch axes
fig.update_xaxes(range=[0, 100], showgrid=False, zeroline=False, visible=False, row=1, col=1)
fig.update_yaxes(range=[0, 100], showgrid=False, zeroline=False, visible=False,
                 scaleanchor="x", scaleratio=0.68, row=1, col=1)

# Ranking axes
fig.update_xaxes(title_text="xT created", row=1, col=2)
fig.update_yaxes(title_text="", autorange="reversed", row=1, col=2)

fig.update_layout(
    paper_bgcolor="white", plot_bgcolor="white",
    width=1200, height=650,
    margin=dict(l=20, r=20, t=80, b=20),
)

# Pitch shapes
add_pitch_shapes(fig, row=1, col=1)

# ---- Pass lines (single colour, no hover) ---
legend_added = False
for _, r in topN.iterrows():
    fig.add_trace(
        go.Scatter(
            x=[r["x0"], r["x1"]], y=[r["y0"], r["y1"]],
            mode="lines",
            line=dict(color="#b70000", width=2),
            hoverinfo="skip",
            legendgroup="pass", showlegend=(not legend_added),
            name="Pass / Cross",
            opacity=0.5,
        ),
        row=1, col=1,
    )
    legend_added = True

# ---- Endpoints (hover with player + xT) ---
fig.add_trace(
    go.Scatter(
        x=topN["x1"], y=topN["y1"],
        mode="markers",
        marker=dict(
            size=topN["xT_added"].clip(0, 0.05) * 800 + 4,
            opacity=0.85,
            color="rgba(120, 140, 255, 0.9)",
        ),
        hovertext=[f"{p}<br>xT+ {v:.4f}" for p, v in zip(topN["player_label"], topN["xT_added"])],
        hoverinfo="text",
        name="Endpoints",
        legendgroup="endpoints",
        showlegend=True,
    ),
    row=1, col=1,
)

# ---- Bar chart (right) ---
fig.add_trace(
    go.Bar(
        x=rank_df["xT_added"], y=rank_df["player_label"],
        orientation="h", marker_color="red",
        showlegend=False,
        hovertemplate="%{y}<br>xT: %{x:.3f}<extra></extra>",
    ),
    row=1, col=2,
)

fig.update_layout(
    title=f"{TEAM_TO_ANALYSE} — Top Positive xT Passes / Crosses",
    legend=dict(x=0.82, y=1.08, xanchor="left", yanchor="top",
                bgcolor="rgba(255,255,255,0.95)"),
)
fig.show()

# =============================================================================
# SINGLE-MATCH xT HEATMAP — where xT is created (by action end location)
# =============================================================================
bins_x, bins_y = 12, 8
xedges = np.linspace(0, 100, bins_x + 1)
yedges = np.linspace(0, 100, bins_y + 1)

H, xe, ye = np.histogram2d(
    prog["x1"], prog["y1"],
    bins=[xedges, yedges],
    weights=prog["xT_added"],
)

fig2 = go.Figure(data=go.Heatmap(
    z=H.T,
    x=(xe[:-1] + xe[1:]) / 2,
    y=(ye[:-1] + ye[1:]) / 2,
    colorscale="Reds",
    hovertemplate="x:%{x:.1f}<br>y:%{y:.1f}<br>xT added:%{z:.4f}<extra></extra>",
))
fig2 = pitch_layout(fig2, f"{TEAM_TO_ANALYSE} — Where xT is created (end location)")
fig2.show()

Success column: outcome  →  kept 436/533 successful actions

Completed passes: 428  |  Positive xT: 347  |  Total xT added: 1.1605


In [36]:
# =============================================================================
# SEASON AGGREGATE xT HEATMAP — current season only, selected team
# =============================================================================
# Scans match CSVs in DATA_DIR (the same folder the single-match file sits in,
# i.e. ONE season).  Pre-filters by filename so only the team's matches are
# loaded — much faster than reading all 230 files.
# =============================================================================

import warnings, re

# ---- 1) Derive a short team name for filename matching ----------------------
# Filenames look like "22_Inter_Pisa_ewod71x...csv" — team names are the 2nd
# and 3rd underscore-delimited tokens.  We extract a "short name" from the
# selected match file to grep for in all filenames.

def _extract_short_team_name(file_name: str, full_team_name: str) -> str:
    """
    Return the short name (e.g. 'Inter') used in filenames, derived from
    FILE_NAME.  Falls back to the first word of the full CSV team name.
    """
    parts = file_name.split("_")
    # parts[0] = matchday, parts[1] = home, parts[2] = away, parts[3+] = id
    if len(parts) >= 4:
        for p in parts[1:3]:            # check home & away tokens
            if p.lower() in full_team_name.lower():
                return p
    # fallback: first word of full name
    return full_team_name.split()[0]

SHORT_NAME = _extract_short_team_name(FILE_NAME, str(TEAM_TO_ANALYSE))
print(f"Team full name : {TEAM_TO_ANALYSE}")
print(f"Filename token : {SHORT_NAME}")

# ---- 2) Pre-filter CSVs by filename ----------------------------------------
all_csvs = sorted(DATA_DIR.glob("*.csv"))
team_csvs = [p for p in all_csvs if f"_{SHORT_NAME}_" in p.name]
print(f"Season files in {DATA_DIR.name}/: {len(all_csvs)} total, "
      f"{len(team_csvs)} contain '{SHORT_NAME}'")

# ---- 3) Loop only over the team's matches ----------------------------------
season_prog_list: list[pd.DataFrame] = []
skipped: list[str] = []

for csv_path in team_csvs:
    try:
        mdf = load_match_csv(csv_path)
    except Exception as exc:
        skipped.append(f"{csv_path.name}: {exc}")
        continue

    # Detect team column
    try:
        t_col = detect_team_col(mdf)
    except KeyError:
        skipped.append(f"{csv_path.name}: no team column")
        continue

    # Verify team is really in this file (safety check)
    teams_in_match = mdf[t_col].dropna().astype(str).unique()
    if str(TEAM_TO_ANALYSE) not in teams_in_match:
        skipped.append(f"{csv_path.name}: '{TEAM_TO_ANALYSE}' not in {list(teams_in_match)}")
        continue

    match_team_df = mdf[mdf[t_col].astype(str) == str(TEAM_TO_ANALYSE)].copy()

    # Column detection (per file, for robustness)
    m_type = _first_existing(mdf.columns,
                ["type_name", "type", "event", "event_type", "eventType", "eventName"])
    m_x  = _first_existing(mdf.columns, ["x", "X", "start_x", "startX", "pos_x"])
    m_y  = _first_existing(mdf.columns, ["y", "Y", "start_y", "startY", "pos_y"])
    m_ex = _first_existing(mdf.columns, ["endX", "end_x", "x_end", "Pass End X", "carry_end_x"])
    m_ey = _first_existing(mdf.columns, ["endY", "end_y", "y_end", "Pass End Y", "carry_end_y"])

    if m_x is None or m_y is None or m_ex is None or m_ey is None:
        skipped.append(f"{csv_path.name}: missing coord columns")
        continue

    # Pass/cross mask (reuse the notebook helper with per-file type column)
    match_passes = match_team_df[guess_pass_mask(match_team_df, type_col=m_type)].copy()
    if match_passes.empty:
        continue

    # Keep only successful actions
    s_col = guess_success_col(match_passes)
    if s_col is not None:
        match_passes = match_passes[mask_success(match_passes, s_col)].copy()

    # Normalise coordinates
    match_passes["x0"] = as_numeric(match_passes[m_x]).clip(0, 100)
    match_passes["y0"] = as_numeric(match_passes[m_y]).clip(0, 100)
    match_passes["x1"] = as_numeric(match_passes[m_ex]).clip(0, 100)
    match_passes["y1"] = as_numeric(match_passes[m_ey]).clip(0, 100)
    match_passes = match_passes.dropna(subset=["x0", "y0", "x1", "y1"])
    if match_passes.empty:
        continue

    # Flip to L→R & compute xT
    match_passes = flip_to_ltr(match_passes)
    match_passes = compute_xt_added(match_passes)

    pos = match_passes[match_passes["xT_added"] > 0]
    if not pos.empty:
        season_prog_list.append(pos[["x1", "y1", "xT_added"]])

if skipped:
    for s in skipped:
        warnings.warn(f"Skipped — {s}")

# ---- 4) Aggregate & plot ----------------------------------------------------
if not season_prog_list:
    print(f"⚠️  No positive-xT actions found for {TEAM_TO_ANALYSE}")
else:
    season_prog = pd.concat(season_prog_list, ignore_index=True)
    n_matches = len(season_prog_list)

    print(f"\n{'='*55}")
    print(f"Season summary for {TEAM_TO_ANALYSE}")
    print(f"  Matches processed : {n_matches}")
    print(f"  Positive xT actions: {len(season_prog)}")
    print(f"  Total xT added    : {season_prog['xT_added'].sum():.4f}")
    print(f"{'='*55}")

    # Heatmap (12×8, weighted by xT_added at end location)
    bins_x, bins_y = 12, 8
    xedges = np.linspace(0, 100, bins_x + 1)
    yedges = np.linspace(0, 100, bins_y + 1)

    H_season, xe, ye = np.histogram2d(
        season_prog["x1"], season_prog["y1"],
        bins=[xedges, yedges],
        weights=season_prog["xT_added"],
    )

    fig_season = go.Figure(data=go.Heatmap(
        z=H_season.T,
        x=(xe[:-1] + xe[1:]) / 2,
        y=(ye[:-1] + ye[1:]) / 2,
        colorscale="Reds",
        hovertemplate="x:%{x:.1f}<br>y:%{y:.1f}<br>xT added:%{z:.4f}<extra></extra>",
    ))

    fig_season = pitch_layout(
        fig_season,
        f"{TEAM_TO_ANALYSE} — Season Aggregate xT Created ({n_matches} matches, end location)",
    )
    fig_season.show()

Team full name : Bologna FC 1909
Filename token : Bologna
Season files in events/: 270 total, 27 contain 'Bologna'

Season summary for Bologna FC 1909
  Matches processed : 27
  Positive xT actions: 8091
  Total xT added    : 28.5565

Season summary for Bologna FC 1909
  Matches processed : 27
  Positive xT actions: 8091
  Total xT added    : 28.5565
